In [12]:
from collections import defaultdict
from dataclasses import dataclass, field
from pathlib import Path
from typing import List
from typing import Tuple
import numpy as np
import pandas as pd
from rdkit import DataStructs, Chem

from typing import Dict
from rdkit.Chem.Scaffolds.MurckoScaffold import MurckoScaffoldSmiles

from rgfn.gfns.reaction_gfn.proxies.path_cost_proxy import PathCostProxy

task_name = 'seh'
table_type = 'missing'

name_map = {
    'rgfn_old': 'RGFN',
    'rxnflow': 'RxnFlow',
    'synflownet': 'SynFlowNet',
    'synflownet_replay': 'SynFlowNet (rt=1.0)',
    'synflownet_replay_2': 'SynFlowNet (rt=0.0)',
    'rgfn_is_decomposable': '\our{} (w/o C)',
    'rgfn_cost_biasing': '\our{} (C)',
    'rgfn_expl_bias': '\our{} (C+P)',
    'rgfn_cost_biasing_2': '\our{} (C)',
    'rgfn_expl_bias_2': '\our{} (C+P)',
    'rgfn_cost_reward_proper': '\our{} (cost reward)',
    'rgfn_48k_cheapest': '\our{} (48k cheapest)',
    'rgfn_56k_cheapest': '\our{} (56k cheapest)',
    'rgfn_96k_cheapest': '\our{} (96k cheapest)',
    'rgfn_112k_cheapest': '\our{} (112k cheapest)',
    'rgfn_400_cheapest': '\our{} (400 cheapest)',
    'rgfn_380_cheapest': '\our{} (380 cheapest)',
    'rgfn_rl_cost': '\our{} (cost RL)',
    'dyn_lib_reward_exploitation_pen_none': '\our{} (C+D+P)',
    'dyn_lib_reward_uniform_policy': '\our{} (C+D)',
    'rgfn_dyn': '\our{} (C+D)',
    'rgfn_dyn_expl': '\our{} (C+D+P)',
    'rgfn_cost_ablation_ml_dataset': 'ML w/ dataset',
    'rgfn_cost_ablation_ml_no_dataset': 'ML w/o dataset',
    'rgfn_cost_ablation_constant_dataset': 'Const. w/ dataset',
    'rgfn_cost_ablation_constant_no_dataset': 'Const. w/o dataset',
    'rgfn_penalty_ablation_0_1': r'$\Delta \tau = 0, \tau_0=1.0$',
    'rgfn_penalty_ablation_0.25_1': r'$\Delta \tau = 0.25, \tau_0=1.0$',
    'rgfn_penalty_ablation_0_2': r'$\Delta \tau = 0, \tau_0=2.0$',
    'rgfn_penalty_ablation_0.25_2': r'$\Delta \tau = 0.25, \tau_0=2.0$',
    'rgfn_penalty_ablation_0_3': r'$\Delta \tau = 0, \tau_0=3.0$',
    'rgfn_penalty_ablation_0.25_3': r'$\Delta \tau = 0.25, \tau_0=3.0$',
    'rgfn_cost_ablation_2_constant_none': 'Constant',
    'rgfn_cost_ablation_2_ml_none': 'ML predicted',
    'rgfn_no_guidance_uniform': 'SCENT (w/o guidance)',
    'rgfn_old_r_2': 'RGFN (r=2)',
    'rgfn_old_r_3': 'RGFN (r=3)',
    'synflownet_r_2': 'SynFlowNet (r=2)',
    'synflownet_r_3': 'SynFlowNet (r=3)',
    'graph_ga': 'GraphGA', 
    'fgfn': 'FGFN', 
    'fgfn_sas': 'FGFN + SA', 
    'reinvent': 'REINVENT',
}
setting_map = {
    'rgfn_new_filtered': "SMALL",
    'synflow_64': "MEDIUM",
    'synflow_128': "LARGE",
    'synflow_2_128': "LARGE",
    'unconstrained': "UNCONSTRAINED",
}

add_missing = False
if table_type == 'synflow':
    threshold = 7.2 if task_name == 'seh' else 0.7
else:
    threshold = 8.0 if task_name == 'seh' else 0.8
n_molecules = 1000

if table_type == 'main':
    templates_name_list = ['rgfn_new_filtered', 'synflow_64', 'synflow_2_128']
    model_names = ['rgfn_old', 'synflownet', 'rxnflow', 'rgfn_is_decomposable', 'rgfn_cost_biasing',
                   'dyn_lib_reward_uniform_policy', 'rgfn_dyn', 'rgfn_expl_bias',
                   'dyn_lib_reward_exploitation_pen_none', 'rgfn_dyn_expl']
elif table_type == 'missing':
    templates_name_list = ['rgfn_new_filtered']
    model_names = ['rxnflow']
elif table_type == 'less':
    templates_name_list = ['synflow_64']
    model_names = ['rgfn_old_r_2', 'rgfn_old_r_3', 'synflownet_r_2', 'synflownet_r_3']
elif table_type == 'final_shit':
    templates_name_list = ['rgfn_new_filtered', 'unconstrained']
    model_names = ['rgfn_old', 'synflownet', 'rxnflow', 'rgfn_is_decomposable', 'rgfn_expl_bias', 'dyn_lib_reward_uniform_policy', 'rgfn_dyn_expl', 'graph_ga', 'fgfn', 'fgfn_sas', 'reinvent']
    model_names = ['rgfn_old', 'synflownet', 'rxnflow', 'rgfn_dyn_expl', 'graph_ga', 'fgfn', 'fgfn_sas', 'reinvent']
    # model_names = ['rgfn_dyn_expl', 'graph_ga', 'fgfn', 'reinvent']
elif table_type == 'synflownet_replay':
    templates_name_list = ['rgfn_new_filtered']
    model_names = ['synflownet', 'synflownet_replay_2', 'synflownet_replay', 'synflownet_replay_free', 'synflownet_replay_uniform']
elif table_type == 'cost_ablation':
    templates_name_list = ['rgfn_new_filtered', 'synflow_64', 'synflow_2_128']
    model_names = ['rgfn_is_decomposable',
                   'rgfn_cost_reward_proper',
                   'rgfn_380_cheapest', 'rgfn_400_cheapest',
                   'rgfn_48k_cheapest', 'rgfn_56k_cheapest',
                   'rgfn_96k_cheapest', 'rgfn_112k_cheapest',
                   'rgfn_cost_biasing']
elif table_type == 'cost_ablation_2':
    templates_name_list = ['rgfn_new_filtered', 'synflow_64', 'synflow_2_128']
    model_names = ['rgfn_cost_ablation_2_constant_none', 'rgfn_cost_ablation_2_ml_none']
elif table_type == 'synflow':
    templates_name_list = ['rgfn_new_filtered', 'synflow_64', 'synflow_2_128']
    model_names = ['rgfn_old', 'synflownet', 'rxnflow', 'rgfn_is_decomposable', 'rgfn_cost_biasing',
                   'dyn_lib_reward_uniform_policy', 'rgfn_expl_bias', 'dyn_lib_reward_exploitation_pen_none']
elif table_type == 'decomposability_ablation':
    templates_name_list = ['rgfn_new_filtered', 'synflow_64']
    model_names = ['rgfn_is_decomposable', 'rgfn_no_guidance_uniform']
elif table_type == 'expl_ablation':
    templates_name_list = ['rgfn_new_filtered', 'synflow_64']
    name_map = {
        'rgfn_is_decomposable_no_exploration': 'dec. disabled',
        'rgfn_is_decomposable': 'dec. uniform',
        'rgfn_is_decomposable_uniform_1000': 'dec. decaying uniform',
        'rgfn_expl_no_linear': 'dec. penalty',
        'rgfn_expl': 'dec. decaying penalty',
        'rgfn_cost_biasing_no_exploration': 'cost. disabled',
        'rgfn_cost_biasing': 'cost. uniform',
        'rgfn_cost_biasing_uniform_1000': 'cost. decaying uniform',
        'rgfn_expl_bias_no_linear': 'cost. penalty',
        'rgfn_expl_bias': 'cost. decaying penalty',
    }
    model_names = [
        'rgfn_is_decomposable_no_exploration',
        'rgfn_is_decomposable',
        'rgfn_is_decomposable_uniform_1000',
        'rgfn_expl_no_linear',
        'rgfn_expl',
        'rgfn_cost_biasing_no_exploration',
        'rgfn_cost_biasing',
        'rgfn_cost_biasing_uniform_1000',
        'rgfn_expl_bias_no_linear',
        'rgfn_expl_bias',
    ]
elif table_type == 'expl_ablation_2':
    model_names = ['rgfn_penalty_ablation_0_1', 'rgfn_penalty_ablation_0.25_1',
                   'rgfn_penalty_ablation_0_2', 'rgfn_penalty_ablation_0.25_2', 'rgfn_penalty_ablation_0_3',
                   'rgfn_penalty_ablation_0.25_3']
    templates_name_list = ['rgfn_new_filtered', 'synflow_64', 'synflow_2_128']
# average reward, average tanimoto similarity (diversity), number of scaffolds, avergage cost, (novelty)

In [13]:
def get_all_chembl_ecfps():
    from rdkit.DataStructs.cDataStructs import CreateFromBitString
    with open('data/all_chembl_ecfps.txt') as f:
        fps = [CreateFromBitString(line.strip()) for line in f]
    print(f"Loaded {len(fps)} fps")
    return fps

In [14]:
# import pandas as pd
# df = pd.read_csv('data/part1.tsv', sep='\t')
# df
# 
# # smiles only dataframe
# pd.DataFrame({'smiles': df['Smiles'].tolist()}).to_csv('data/all_chembl.csv')
# 
# from tqdm import tqdm
# from notebooks.utils.common import _get_fp
# from rdkit import Chem
# from rdkit.Chem import AllChem
# from rdkit import DataStructs
# from rdkit.DataStructs.cDataStructs import CreateFromBitString
# 
# fps = []
# for smiles in tqdm(df['Smiles'], total=len(df)):
#     if isinstance(smiles, str):
#         fp = _get_fp(smiles, "morgan_2", n_bits=1024)
#         if fp is not None:
#             fps.append(fp)
#         
# unique_fps = list({fp.ToBitString() for fp in fps})
# with open('data/all_chembl_ecfps.txt', 'w') as f:
#     for bs in unique_fps:
#         f.write(f"{bs}\n")

In [15]:
from notebooks.utils.sampling_results import SamplingResult, get_average_reward, get_average_tanimoto_similarity, \
    get_num_scaffolds, SamplingResultsList, get_diversity, get_novelty
from notebooks.utils.common import get_path_costs, get_path_cost_proxy
from notebooks.utils.training_results import TrainingResults, TrainingResultsList

all_chembl_ecfps = None
results_dict = {}
training_results_dict = {}
for templates_name in templates_name_list:
    path_cost_proxy = None
    sanitized_path_cost_proxy = None
    rxnflow_path_cost_proxy = None
    results_list = []
    training_results_list = []
    for model_name in model_names:
        sub_list = []
        training_sub_list = []
        for i in range(5):
            try:
                result = SamplingResult(model_name=model_name, seed=i, templates_name=templates_name,
                                        task_name=task_name, threshold=threshold, include_paths=False)
                training_results = TrainingResults(model_name=model_name, templates_name=templates_name,
                                                   task_name=task_name, threshold=threshold, seed=i, include_paths=not templates_name == 'unconstrained')
            except FileNotFoundError as f:
                print(f"Skipping {model_name} {i}, {f}")
                continue
            if result.average_reward is None:
                result.average_reward = get_average_reward(result)
            if result.average_tanimoto_similarity is None:
                result.average_tanimoto_similarity = get_average_tanimoto_similarity(result)
            if result.num_scaffolds is None:
                result.num_scaffolds = get_num_scaffolds(result)
            if result.novelty is None:
                all_chembl_ecfps = all_chembl_ecfps or get_all_chembl_ecfps()
                result.novelty = get_novelty(result, all_chembl_ecfps, top_n=100)
            if result.diversity is None:
                result.diversity = get_diversity(result, top_n=100)
            # if result.average_cost is None and templates_name != 'unconstrained':
            #     if 'synflow' in result.model_name:
            #         sanitized_path_cost_proxy = sanitized_path_cost_proxy or get_path_cost_proxy(templates_name,
            #                                                                                      sanitized=True)
            #         actual_path_cost_proxy = sanitized_path_cost_proxy
            #     elif 'rxnflow' in result.model_name:
            #         rxnflow_path_cost_proxy = rxnflow_path_cost_proxy or get_path_cost_proxy(templates_name,
            #                                                                                  sanitized="rxnflow")
            #         actual_path_cost_proxy = rxnflow_path_cost_proxy
            #     else:
            #         path_cost_proxy = path_cost_proxy or get_path_cost_proxy(templates_name, sanitized=False)
            #         actual_path_cost_proxy = path_cost_proxy
            #     if not hasattr(training_results, 'molecule_to_cheapest_cost'):
            #         training_results.molecule_to_cheapest_cost = None
            # 
            #     costs, _ = get_path_costs(result, actual_path_cost_proxy, training_results.molecule_to_cheapest_cost)
            #     result.average_cost = costs.mean()
            result.save()
            sub_list.append(result)
            training_sub_list.append(training_results)
        if sub_list:
            results_list.append(SamplingResultsList(sub_list))
            training_results_list.append(TrainingResultsList(training_sub_list))

    results_dict[templates_name] = results_list
    training_results_dict[templates_name] = training_results_list


Skipping rxnflow 0, File /Volumes/External Disk/RGFN/notebooks/results/rgfn_new_filtered/seh/rxnflow/sampling/molecules_0.csv not found.


tanimoto_similarities: 0it [00:00, ?it/s]
/Users/piotrgainski/Desktop/projects/RGFN/notebooks/utils/sampling_results.py:134: RuntimeWarning: Mean of empty slice.
  return np.array(tanimoto_similarities).mean()
/Users/piotrgainski/miniconda3/envs/rgfn/lib/python3.11/site-packages/numpy/core/_methods.py:129: RuntimeWarning: invalid value encountered in scalar divide
  ret = ret.dtype.type(ret / rcount)


Loaded 1801618 fps


diversity: 100%|██████████| 100/100 [00:00<00:00, 53759.34it/s]
tanimoto_similarities: 0it [00:00, ?it/s]
/Users/piotrgainski/Desktop/projects/RGFN/notebooks/utils/sampling_results.py:134: RuntimeWarning: Mean of empty slice.
  return np.array(tanimoto_similarities).mean()
/Users/piotrgainski/miniconda3/envs/rgfn/lib/python3.11/site-packages/numpy/core/_methods.py:129: RuntimeWarning: invalid value encountered in scalar divide
  ret = ret.dtype.type(ret / rcount)
diversity: 100%|██████████| 100/100 [00:00<00:00, 39184.45it/s]


Skipping rxnflow 3, File not found: /Volumes/External Disk/RGFN/notebooks/results/rgfn_new_filtered/seh/rxnflow/training/paths_3.csv
Skipping rxnflow 4, File /Volumes/External Disk/RGFN/notebooks/results/rgfn_new_filtered/seh/rxnflow/sampling/molecules_4.csv not found.


In [16]:
!pwd

/Users/piotrgainski/Desktop/projects/RGFN/notebooks


In [18]:
def prettify_mean_std(mean: float, std: float, add_std: bool = True):
    decimals = 2
    if mean >= 10:
        decimals = 1
    if mean >= 100:
        decimals = 0

    if decimals == 0:
        if add_std:
            return f"{int(mean)} \std{{{int(std)}}}"
        else:
            return f"{int(mean)} ± {int(std)}"
    if add_std:
        return f"{round(mean, decimals)} \std{{{round(std, decimals)}}}"
    else:
        return f"{round(mean, decimals)} ± {round(std, decimals)}"


def bold_best_with_std(df):
    def get_value(x):
        if '\std' in x:
            return float(x.split('\std')[0])
        return float('nan')

    for col in df.columns[1:]:  # Skip the "Name" column
        # Extract mean values as floats for comparison
        print(col)
        means = df[col].str.extract(r"([0-9.]+)").astype(float)[0]
        if 'sim' in col or 'cost' in col:
            max_mean = means.min()
        else:
            max_mean = means.max()

        # Apply bold formatting to rows with the maximum mean
        df[col] = df[col].apply(
            lambda x: f"\\textbf{{{x}}}" if get_value(x) == max_mean else x
        )
    return df


dfs = []
for templates_name, results_list in results_dict.items():
    training_results_list = training_results_dict[templates_name]
    if add_missing:
        missing_models = list(set(model_names) - set([r.model_name for r in results_list]))
    else:
        missing_models = []
    missing_results = ['-'] * len(missing_models)
    df = pd.DataFrame({
        'model': missing_models + [result.model_name for result in results_list],
        f'modes $>$ {threshold} $\\uparrow$': missing_results + [prettify_mean_std(*r.get_num_modes_mean_std()) for r in
                                                                 training_results_list],
        f'scaff. $>$ {threshold} $\\uparrow$': missing_results + [prettify_mean_std(*r.get_num_scaffolds_mean_std())
                                                                   for r in
                                                                   training_results_list],
        'reward $\\uparrow$': missing_results + [prettify_mean_std(*r.get_reward_mean_std()) for r in results_list],
        # f'sim. $>$ {threshold} $\\downarrow$': missing_results + [
        #     prettify_mean_std(*r.get_tanimoto_similarity_mean_std()) for r in
        #     results_list],
        # f'scaff. $>$ {threshold} $\\uparrow$': missing_results + [prettify_mean_std(*r.get_num_scaffolds_mean_std()) for
        #                                                           r in
        #                                                           results_list],
        'novelty': missing_results + [prettify_mean_std(*r.get_novelty_mean_std()) for r in results_list],
        'diversity': missing_results + [prettify_mean_std(*r.get_diversity_mean_std()) for r in results_list],
        # f'cost $>$ {threshold} $\\downarrow$': missing_results + [prettify_mean_std(*r.get_average_cost_mean_std()) for
        #                                                           r in
        #                                                           results_list],
    })
    df = bold_best_with_std(df)
    df.insert(0, 'setting', setting_map[templates_name])
    df['model'] = df['model'].map(name_map)
    # df = df.applymap(lambda x: x.replace('_', ' '))
    df.columns = df.columns.str.replace('_', ' ')
    dfs.append(df)
df = pd.concat(dfs)
df

modes $>$ 8.0 $\uparrow$
scaff. $>$ 8.0 $\uparrow$
reward $\uparrow$
novelty
diversity


,setting,model,modes $>$ 8.0 $\uparrow$,scaff. $>$ 8.0 $\uparrow$,reward $\uparrow$,novelty,diversity
0,SMALL,RxnFlow,\textbf{7.5 \std{1.5}},\textbf{11.5 \std{1.5}},\textbf{6.11 \std{0.02}},\textbf{0.53 \std{0.01}},\textbf{0.8 \std{0.0}}


In [ ]:
import pyperclip


def group_rows_to_latex(df, group_col, caption=""):
    grouped = df.groupby(group_col, sort=False)
    rows = []

    # Dynamically infer column names from the DataFrame
    columns = df.columns.tolist()

    for group, data in grouped:
        # Add the multirow command for the group

        group_row = f"\\multirow{{{len(data)}}}{{*}}{{\\rotatebox[origin=c]{{90}}{{{group}}}}}"
        first = True
        rows.append(r"\midrule")
        for _, row in data.iterrows():
            rest_columns = ' & '.join([row[col] for col in columns[1:]])
            if first:
                # Add the group name to the first row
                rows.append(f"{group_row} & {rest_columns} \\\\")
                first = False
            else:
                # Add only the data for subsequent rows, with a placeholder for the group column
                rows.append(f" & {rest_columns} \\\\")

    # Combine rows into a LaTeX table
    table = (
            r"\begin{table*}[t]" "\n"
            r"\centering" "\n"
            f"\caption{{{caption}}}" "\n"
            f"\label{{tab:{table_type}_{task_name}}}" "\n"
            r"\begin{tabular}{" + "c" * len(columns) + "}" "\n"  # Dynamic column specifier
                                                       r"\toprule" "\n"
                                                       r"\multirow{2}{*}{} & \multirow{2}{*}{model} & \multicolumn{2}{c}{online discovery} & \multicolumn{2}{c}{inference sampling} \\" "\n"
                                                       r"\cmidrule(lr{1em}){3-4}" "\n"
                                                       r"\cmidrule(l{1em}){5-6}" "\n"
                                                       r"& & " + "& ".join(df.columns[2:8]) + r" \\" "\n"
            + '\n'.join(rows) + "\n"  # Using join without f-string to avoid escape issues
                                r"\bottomrule" "\n"
                                r"\end{tabular}" "\n"
                                r"\end{table*}"
    )
    return table


if table_type == 'main':
    caption = f'Evaluation of different template-based approaches using \\{task_name}{{}} proxy in SMALL, MEDIUM and LARGE settings. We observe that \our{{}} (C) that uses cost guidance can greatly reduce the cost of the sampled molecules, while drastically increasing the number of scaffolds and modes discovered during the training (in MEDIUM and LARGE settings). Dynamic library (C+D) significantly improves the performance of the model in LOW setting. Using exploitation penalty (+P) further boosts the results, making \our{{}} by far the most powerful evaluated method in all settings.'
elif table_type == 'synflow':
    caption = f'Evaluation of different template-based approaches using \\{task_name}{{}} proxy in SMALL, MEDIUM and LARGE settings, but using lower reward threshold equivalent to the one used in SynFlowNet paper \\cite{{cretu2024synflownet}}. We observe that \our{{}} (C) that uses cost guidance can greatly reduce the cost of the sampled molecules, while drastically increasing the number of scaffolds and modes discovered during the training (in MEDIUM and LARGE settings). Dynamic library (C+D) significantly improves the performance of the model in LOW setting. Using exploitation penalty (+P) further boosts the results, making \our{{}} by far the most powerful evaluated method in all settings. Importantly, the results seem to be consistent with those reported in \\cite{{cretu2024synflownet}}.'
else:
    caption = "Table"

latex_table = group_rows_to_latex(df, 'setting', caption=caption).replace("nan \std{nan}", "-")
pyperclip.copy(latex_table)
print("LaTeX table copied to clipboard!")
print(latex_table)